# 1. Imports

In [51]:
from pathlib import Path
from collections import Counter
import json
import re
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

DATASET_NAME = "mhardalov/exams"
CONFIG_NAME = "multilingual"
TARGET_LANGUAGES = {
    "Italian": "italian",
    "Spanish": "spanish",
}

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUT_DIR = REPO_ROOT / "data" / "processed" / "exams"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUT_DIR)

Repo root: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
Output dir: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/exams


In [41]:
ds = load_dataset(DATASET_NAME, CONFIG_NAME)

print(ds)
print("Splits:", list(ds.keys()))

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 7961
    })
    validation: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 2672
    })
    test: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 13510
    })
})
Splits: ['train', 'validation', 'test']


In [42]:
# flatten rows into a df
rows = []
for split_name in ds.keys():
    for ex in ds[split_name]:
        info = ex.get("info", {}) or {}
        question = ex.get("question", {}) or {}
        choices = question.get("choices", {}) or {}

        labels = choices.get("label", []) or []
        texts = choices.get("text", []) or []
        paras = choices.get("para", []) or []

        rows.append({
            "source_split": split_name,
            "source_id": ex.get("id"),
            "language_raw": info.get("language"),
            "subject": info.get("subject"),
            "grade": info.get("grade"),
            "stem": question.get("stem"),
            "labels": labels,
            "texts": texts,
            "paras": paras,
            "answerKey": ex.get("answerKey"),
            "n_choices": len(labels),
        })

df = pd.DataFrame(rows)

print(f"Total rows: {len(df)}")
print("\nLanguage distribution:")
print(df["language_raw"].value_counts(dropna=False).to_string())

Total rows: 24143

Language distribution:
language_raw
Bulgarian           2937
Croatian            2879
Vietnamese          2443
Hungarian           2267
North Macedonian    2075
Polish              1971
Turkish             1964
Serbian             1637
Albanian            1505
Italian             1256
Portuguese           924
Lithuanian           593
German               577
Arabic               562
French               318
Spanish              235


# 2. Inspect

In [43]:
df_target = df[df["language_raw"].isin(TARGET_LANGUAGES.keys())].copy()
df_target["language"] = df_target["language_raw"].map(TARGET_LANGUAGES)

print(f"Target-language rows: {len(df_target)}")
print(df_target["language"].value_counts(dropna=False).to_string())

print("\nSplit counts:")
print(df_target.groupby(["language", "source_split"]).size().to_string())

print("\nChoice-count distribution:")
print(df_target.groupby(["language", "n_choices"]).size().to_string())

print("\nAnswer-key distribution:")
print(df_target["answerKey"].value_counts(dropna=False).to_string())

Target-language rows: 1491
language
italian    1256
spanish     235

Split counts:
language  source_split
italian   test            636
          train           464
          validation      156
spanish   test            235

Choice-count distribution:
language  n_choices
italian   3             136
          4            1113
          5               7
spanish   3             197
          4              38

Answer-key distribution:
answerKey
B    410
C    396
A    386
D    298
E      1


# 3. Cleaning

## helpers

In [44]:
BAD_TEXT_VALUES = {"", "nan", "none", "null", "n/a", "na", "?", "-", "undefined"}
CORRUPTION_RE = re.compile(r"[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]")
ARTIFACT_RE = re.compile(r"(Unnamed:\s*\d+|\[object Object\])", re.IGNORECASE)
TIMESTAMP_RE = re.compile(r"\b\d{4}-\d{2}-\d{2}\s+00:00:00\b")


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip()).lower()


def is_bad_text(x):
    if x is None:
        return True

    text = str(x).strip()
    low = text.lower()

    if low in BAD_TEXT_VALUES:
        return True
    if CORRUPTION_RE.search(text):
        return True
    if ARTIFACT_RE.search(text):
        return True
    if TIMESTAMP_RE.search(text):
        return True

    return False


def has_nonempty_para(paras):
    if not isinstance(paras, list):
        return False
    return any(str(p).strip() for p in paras if p is not None)


def row_key(row):
    return (
        normalize_text(row["stem"]),
        tuple(normalize_text(x) for x in row["texts"]),
    )


def full_row_key(row):
    return row_key(row) + (str(row["answerKey"]).strip(),)


def validate_row(row):
    labels = row["labels"]
    texts = row["texts"]
    answer = row["answerKey"]

    if row["language"] not in TARGET_LANGUAGES.values():
        return False, "not_target_language"

    if row["source_split"] not in {"train", "validation", "test"}:
        return False, "bad_split"

    if is_bad_text(row["stem"]):
        return False, "bad_stem"

    if not isinstance(labels, list) or not isinstance(texts, list):
        return False, "bad_choices_type"

    if len(labels) != len(texts):
        return False, "label_text_length_mismatch"

    if len(labels) < 2:
        return False, "too_few_choices"

    if answer not in labels:
        return False, "answer_not_in_labels"

    if any(is_bad_text(t) for t in texts):
        return False, "bad_choice_text"

    if len(set(str(l).strip() for l in labels)) != len(labels):
        return False, "duplicate_labels"

    if len(set(normalize_text(t) for t in texts)) != len(texts):
        return False, "duplicate_choice_texts"

    return True, None

## clean/deduplicate

In [45]:
valid_rows = []
rejected_rows = []
stats = Counter()

for _, row in df_target.iterrows():
    ok, reason = validate_row(row)
    row_dict = row.to_dict()

    if ok:
        valid_rows.append(row_dict)
    else:
        row_dict["rejection_reason"] = reason
        rejected_rows.append(row_dict)
        stats[f"removed_{reason}"] += 1

print(f"Initially valid rows: {len(valid_rows)}")
print(f"Initially rejected rows: {len(rejected_rows)}")

# remove conflicting duplicates: same stem/options but different answer.
answers_by_key = {}
for row in valid_rows:
    key = row_key(row)
    answers_by_key.setdefault(key, set()).add(str(row["answerKey"]).strip())

conflicting_keys = {k for k, answers in answers_by_key.items() if len(answers) > 1}

nonconflicting_rows = []
for row in valid_rows:
    if row_key(row) in conflicting_keys:
        row["rejection_reason"] = "conflicting_duplicate"
        rejected_rows.append(row)
        stats["removed_conflicting_duplicate"] += 1
    else:
        nonconflicting_rows.append(row)

# Remove exact duplicates by stem/options/answer.
seen = set()
clean_rows = []

for row in nonconflicting_rows:
    key = full_row_key(row)
    if key in seen:
        row["rejection_reason"] = "exact_duplicate"
        rejected_rows.append(row)
        stats["removed_exact_duplicate"] += 1
    else:
        seen.add(key)
        clean_rows.append(row)

clean = pd.DataFrame(clean_rows)
rejected = pd.DataFrame(rejected_rows)

print(f"Clean rows: {len(clean)}")
print(f"Rejected rows: {len(rejected)}")

print("\nRemoval reasons:")
for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

print("\nClean counts by language/split:")
if len(clean):
    print(clean.groupby(["language", "source_split"]).size().to_string())

print("\nClean choice-count distribution:")
if len(clean):
    print(clean.groupby(["language", "n_choices"]).size().to_string())

Initially valid rows: 1485
Initially rejected rows: 6
Clean rows: 1485
Rejected rows: 6

Removal reasons:
removed_bad_choice_text: 1
removed_duplicate_choice_texts: 5

Clean counts by language/split:
language  source_split
italian   test            634
          train           462
          validation      154
spanish   test            235

Clean choice-count distribution:
language  n_choices
italian   3             136
          4            1107
          5               7
spanish   3             197
          4              38


In [46]:
# format examples into processed MCQ JSONL schema

def format_prompt(stem, labels, texts):
    stem = str(stem).strip()
    option_lines = [
        f"{str(label).strip()}) {str(text).strip()}"
        for label, text in zip(labels, texts)
    ]
    return stem + "\n\n" + "\n".join(option_lines)


def format_jsonl_example(row, idx):
    language = row["language"]
    labels = row["labels"]
    texts = row["texts"]
    answer = str(row["answerKey"]).strip()

    metadata = {
        "source_id": row.get("source_id", ""),
        "source_split": row.get("source_split", ""),
        "language_raw": row.get("language_raw", ""),
        "subject": row.get("subject", ""),
        "grade": row.get("grade", ""),
        "n_choices": len(labels),
        "has_para": has_nonempty_para(row.get("paras", [])),
    }

    return {
        "id": f"exams_{language}_{idx}",
        "dataset": "exams",
        "language": language,
        "task_type": "mcq",
        "prompt": format_prompt(row["stem"], labels, texts),
        "answer": answer,
        "metadata": metadata,
    }

In [47]:
jsonl_examples_by_language = {}

for language in TARGET_LANGUAGES.values():
    lang_df = clean[clean["language"] == language].copy()
    examples = []

    for idx, (_, row) in enumerate(lang_df.iterrows()):
        examples.append(format_jsonl_example(row.to_dict(), idx))

    jsonl_examples_by_language[language] = examples
    print(f"{language}: {len(examples)} JSONL examples")

italian: 1250 JSONL examples
spanish: 235 JSONL examples


In [48]:
def to_csv_safe(df_in):
    df_out = df_in.copy()

    for col in ["labels", "texts", "paras"]:
        if col in df_out.columns:
            df_out[col] = df_out[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, list)
                else x
            )

    return df_out


clean_csv = to_csv_safe(clean)
rejected_csv = to_csv_safe(rejected)

clean_csv.to_csv(OUT_DIR / "exams_clean_all.csv", index=False)
rejected_csv.to_csv(OUT_DIR / "exams_rejected_all.csv", index=False)

for language in TARGET_LANGUAGES.values():
    lang_clean_csv = clean_csv[clean_csv["language"] == language].copy()
    lang_clean_csv.to_csv(OUT_DIR / f"exams_clean_{language}.csv", index=False)

    jsonl_path = OUT_DIR / f"exams_{language}_all.jsonl"
    with jsonl_path.open("w", encoding="utf-8") as f:
        for ex in jsonl_examples_by_language[language]:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    print(f"Saved {language}:")
    print(f"  CSV:   {OUT_DIR / f'exams_clean_{language}.csv'}")
    print(f"  JSONL: {jsonl_path}")

print("\nSaved:")
print(OUT_DIR / "exams_clean_all.csv")
print(OUT_DIR / "exams_rejected_all.csv")

Saved italian:
  CSV:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_clean_italian.csv
  JSONL: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_italian_all.jsonl
Saved spanish:
  CSV:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_clean_spanish.csv
  JSONL: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_spanish_all.jsonl

Saved:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_clean_all.csv
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_rejected_all.csv


In [49]:
# Cell 9 — Quality inspection

N_EXAMPLES = 3

for language, examples in jsonl_examples_by_language.items():
    print("=" * 80)
    print(f"Examples for {language}: {len(examples)} examples")

    for ex in examples[:N_EXAMPLES]:
        print("-" * 80)
        print(f"id: {ex['id']}")
        print(f"answer: {ex['answer']}")
        print(f"metadata: {ex['metadata']}")
        print(ex["prompt"])

Examples for italian: 1250 examples
--------------------------------------------------------------------------------
id: exams_italian_0
answer: D
metadata: {'source_id': '40220e8a-8d4a-11ea-ae7d-54bef70b159e', 'source_split': 'train', 'language_raw': 'Italian', 'subject': 'Biology', 'grade': 12, 'n_choices': 4, 'has_para': False}
Attraverso l’osservazione di un organismo si è visto che le sue cellule contengono il nucleo, i mitocondri ed i cloroplasti. Ai livelli di organizzazione superiori si è costatato che non sono presenti né tessuti né organi. A quale degli organismi elencati di seguito corrisponde tale descrizione?

A) il politrico
B) il paramecio
C) la verongia
D) l’euglena
--------------------------------------------------------------------------------
id: exams_italian_1
answer: D
metadata: {'source_id': '40220e8c-8d4a-11ea-ae7d-54bef70b159e', 'source_split': 'train', 'language_raw': 'Italian', 'subject': 'Biology', 'grade': 12, 'n_choices': 4, 'has_para': False}
Quale dei se

In [50]:
print("=== SUMMARY ===")
print(f"Total rows in full dataset:       {len(df)}")
print(f"Target-language rows:             {len(df_target)}")
print(f"Clean rows:                       {len(clean)}")
print(f"Rejected rows:                    {len(rejected)}")

print("\nClean rows by language:")
if len(clean):
    print(clean["language"].value_counts().to_string())

print("\nClean rows by language/split:")
if len(clean):
    print(clean.groupby(["language", "source_split"]).size().to_string())

print("\nRejected reasons:")
for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

print("\nSaved JSONL files:")
for language in TARGET_LANGUAGES.values():
    path = OUT_DIR / f"exams_{language}_all.jsonl"
    print(path, "exists =", path.exists())

=== SUMMARY ===
Total rows in full dataset:       24143
Target-language rows:             1491
Clean rows:                       1485
Rejected rows:                    6

Clean rows by language:
language
italian    1250
spanish     235

Clean rows by language/split:
language  source_split
italian   test            634
          train           462
          validation      154
spanish   test            235

Rejected reasons:
removed_bad_choice_text: 1
removed_duplicate_choice_texts: 5

Saved JSONL files:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_italian_all.jsonl exists = True
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/multilingual/data/processed/exams/exams_spanish_all.jsonl exists = True
